# Zero-Shot Speaker Diarization with pyannote.audio

This notebook evaluates a pretrained pyannote speaker diarization pipeline on consultation audio from the Primock57 dataset. It first runs inference on a single file, compares the prediction with TextGrid ground truth, and then scales the same workflow to the full audio folder to calculate Diarization Error Rate (DER).

## Environment Setup

Install the pyannote and utility dependencies needed for zero-shot diarization. These cells are intended for a fresh Colab runtime.

In [ ]:
!pip install pyannote.audio --quiet
!pip install gdown --quiet

In [ ]:
pip install pydub

## Select and Preview a Test Recording

Choose one consultation recording for a quick sanity check. Playing the audio before inference helps confirm that the path is correct and the file is readable.

In [ ]:
# Optional test audio
from pydub import AudioSegment
import IPython

audio_file = '/content/drive/MyDrive/Speech_assignment/audio/day1_consultation01_doctor.wav'

IPython.display.Audio(audio_file)

## Run the Pretrained Diarization Pipeline

Load `pyannote/speaker-diarization-3.1` and apply it to the selected audio file. The Hugging Face token is read from the `HF_TOKEN` environment variable so the credential is not stored in the notebook.

In [ ]:
from pyannote.audio import Pipeline
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Set the HF_TOKEN environment variable before loading pyannote models.")

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=HF_TOKEN)

# import torch
# torch.__version__
# pipeline.to(torch.device("cuda"))

# apply pretrained pipeline
diarization = pipeline(audio_file, num_speakers=1)
# with open("audio_1.rttm", "w") as rttm:
#     diarization.write_rttm(rttm)

# # print the result
# for turn, _, speaker in diarization.itertracks(yield_label=True):
#     print(f"start={turn.start:.1f}s stop={turn.end:.1f}s speaker_{speaker}")

## Inspect Predicted Speaker Segments

Print the start and stop time of each predicted speech segment from the diarization output.

In [ ]:
for turn, _, speaker in diarization.itertracks(yield_label=True):
    print(f"start={turn.start:.1f}s stop={turn.end:.1f}s")

## Install Evaluation Dependencies

Install packages used for reading TextGrid annotations and computing diarization metrics.

In [ ]:
!pip install praat-parselmouth pyannote.metrics
!pip install pyannote.parser


In [ ]:
pip install textgrid


## Load TextGrid Ground Truth

Convert a TextGrid speaker tier into a pyannote `Annotation`. Each non-empty interval becomes a labeled time segment that can be compared with model output.

In [ ]:
from textgrid import TextGrid
from pyannote.core import Annotation, Segment

def load_textgrid_as_annotation(textgrid_path, tier_name="Doctor"):
    # Load the TextGrid file
    tg = TextGrid.fromFile(textgrid_path)

    # Initialize an Annotation object
    annotation = Annotation()

    # Find the specified tier
    for tier in tg.tiers:
        if tier.name == tier_name:
            # Extract intervals and create pyannote Segments with speaker labels
            for interval in tier.intervals:
                start_time = interval.minTime
                end_time = interval.maxTime
                text = interval.mark  # The text label in the interval

                # Only add intervals with non-empty labels
                if text.strip():  # Skips empty text intervals
                    annotation[Segment(start_time, end_time)] = tier_name
            break
    else:
        raise ValueError(f"Tier '{tier_name}' not found in TextGrid")

    return annotation

# Load the ground truth TextGrid as an Annotation
reference = load_textgrid_as_annotation("/content/drive/MyDrive/Speech_assignment/transcripts/day1_consultation01_doctor.TextGrid")


## View the Reference Annotation

Display the ground-truth annotation loaded from the transcript file.

In [ ]:
reference

## View the Model Hypothesis

Display the diarization result generated by the pretrained pyannote pipeline.

In [ ]:
diarization

## Calculate DER for the Sample File

Compare the TextGrid reference annotation with the model hypothesis using `DiarizationErrorRate`.

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate

# Assuming `hypothesis` is the diarization result from your model
der_metric = DiarizationErrorRate()
der = der_metric(reference, diarization)
print(f"Diarization Error Rate (DER): {der * 100:.2f}%")

## Optional TextGrid to RTTM Conversion

The TextGrid transcript intervals can also be exported to an RTTM-style text file. This is useful when comparing with tools that expect diarization annotations in RTTM format.

In [ ]:
from textgrid import TextGrid

def textgrid_to_rttm(textgrid_path, rttm_path, speaker_id="Speaker_1"):
    # Load the TextGrid file
    tg = TextGrid.fromFile(textgrid_path)

    # Open RTTM file for writing
    with open(rttm_path, 'w') as rttm_file:
        # Loop through each tier in TextGrid
        for tier in tg.tiers:
            # Process intervals in the tier
            for interval in tier.intervals:
                # Only process intervals with non-empty text (non-silent segments)
                if interval.mark.strip():
                    # RTTM format: SPEAKER <file_id> <channel> <start_time> <duration> <NA> <NA> <speaker_id> <NA> <NA>
                    start_time = interval.minTime
                    duration = interval.maxTime - interval.minTime
                    rttm_line = f"start={start_time:.3f}s stop={duration:.3f}s"
                    rttm_file.write(rttm_line + '\n')

    print(f"RTTM file created at: {rttm_path}")

# Usage example:
textgrid_to_rttm(
    textgrid_path="/content/drive/MyDrive/Speech_assignment/transcripts/day1_consultation01_doctor.TextGrid",
    rttm_path="/content/gt.rttm",
    speaker_id="Doctor"  # Specify your speaker name or ID
)


## Install Metric Utilities

Ensure the core pyannote annotation classes and metric package are available before running the full-dataset evaluation.

In [ ]:
pip install pyannote.core pyannote.metrics


## Run Diarization Across the Dataset

Apply the pretrained pipeline to each `.wav` file in the audio directory and store the predictions by filename for later metric calculation.

In [ ]:
from pyannote.audio import Pipeline
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Set the HF_TOKEN environment variable before loading pyannote models.")

# Initialize the diarization pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=HF_TOKEN)

# Define the path to your audio files
audio_files_path = "/content/drive/MyDrive/Speech_assignment/audio/"  # Replace with the path to your audio files

# Dictionary to store diarization results for each file
diarization_results = {}

# Loop through each audio file
for audio_file in os.listdir(audio_files_path):
    if audio_file.endswith(".wav"):  # Check for your specific audio file extension
        audio_file_path = os.path.join(audio_files_path, audio_file)

        # Perform diarization
        diarization = pipeline(audio_file_path, num_speakers=1)
        print(audio_file)

        # Store the result in the dictionary
        diarization_results[audio_file] = diarization

# Now you have all diarization results in 'diarization_results' dictionary
# You can access each result by the filename, e.g., diarization_results['file_name.wav']
print("Diarization complete for all files.")

## Check the Number of Processed Files

Print the number of audio files that produced diarization results.

In [ ]:
print(len(diarization_results))

## Evaluate Full-Dataset DER

Load each TextGrid transcript, select the correct speaker tier based on the file name, align it with the matching diarization output, and calculate per-file DER plus the average DER.

In [ ]:
from textgrid import TextGrid
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate
import os

def load_textgrid_as_annotation(textgrid_path, tier_priority):
    # Load the TextGrid file
    tg = TextGrid.fromFile(textgrid_path)

    # Initialize an Annotation object
    annotation = Annotation()

    # Find the specified tier in priority order
    found_tier = None
    for tier_name in tier_priority:
        for tier in tg.tiers:
            if tier.name == tier_name:
                found_tier = tier
                break
        if found_tier:
            break

    if not found_tier:
        print(f"Warning: No matching tier found in {textgrid_path}")
        return None

    # Extract intervals and create pyannote Segments with speaker labels
    for interval in found_tier.intervals:
        start_time = interval.minTime
        end_time = interval.maxTime
        text = interval.mark  # The text label in the interval

        # Only add intervals with non-empty labels
        if text.strip():  # Skips empty text intervals
            annotation[Segment(start_time, end_time)] = found_tier.name

    return annotation

# Initialize Diarization Error Rate (DER) metric
der_metric = DiarizationErrorRate()

# Path to your .TextGrid files
textgrid_files_path = "/content/drive/MyDrive/Speech_assignment/transcripts/"

# Calculate DER for each file
der_results = {}

for textgrid_file in os.listdir(textgrid_files_path):
    if textgrid_file.endswith(".TextGrid"):
        # Determine priority of tier names based on file name
        if "doctor" in textgrid_file.lower():
            tier_priority = ["Doctor", "Speaker"]
        elif "patient" in textgrid_file.lower():
            tier_priority = ["Patient", "Speaker"]
        else:
            tier_priority = ["Speaker"]

        # Get the audio file name (assuming naming consistency between audio and TextGrid)
        audio_file_name = textgrid_file.replace(".TextGrid", ".wav")

        # Load the TextGrid as reference annotation
        textgrid_path = os.path.join(textgrid_files_path, textgrid_file)
        reference = load_textgrid_as_annotation(textgrid_path, tier_priority)

        if reference is None:
            # Skip this file if no suitable tier is found
            continue

        # Get the diarization result from the precomputed results
        hypothesis = diarization_results.get(audio_file_name)

        if hypothesis:
            # Compute DER for this file and store it
            der = der_metric(reference, hypothesis)
            der_results[audio_file_name] = der
            print(f"DER for {audio_file_name}: {der:.4f}")
        # else:
        #     print(f"No diarization result for {audio_file_name}")

# Calculate average DER
if der_results:
    average_der = sum(der_results.values()) / len(der_results)
    print(f"\nAverage DER for all files: {average_der*100 :.4f}")
else:
    print("No DER calculated; check data consistency.")


## Baseline Result

The zero-shot pretrained `pyannote/speaker-diarization-3.1` pipeline produced an average DER of **21.8011%** in this evaluation run.

In [ ]:
if der_results:
    average_der = sum(der_results.values()) / len(der_results)
    print(f"\nAverage DER for all files: {average_der*100 :.4f}")
else:
    print("No DER calculated; check data consistency.")